#  Introduction:
    Student Name: Inon R.

    Last 4 digits of mt ID: 1787.

# AI Prompting Documentation:
        - what is the optimal lerning reat for the model
        - is ther a bether way to implement the KNN model
        - how to clculet the gradient

# Data set info
        Dataset separated in two files:
            Fake.csv (23502 fake news article)
            True.csv (21417 true news article)

        Dataset columns:
            Title: title of news article
            Text: body text of news article
            Subject: subject of news article

        Date: publish date of news article
        link: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset

In [1]:
# Importing necessary libraries for data manipulation and text processing.
import pandas as pd
import numpy as np
import re
import string



In [2]:
# Loading the datasets, assigning binary target labels (0=Fake, 1=True), and merging them into one dataframe.
df_fake = pd.read_csv("data/Fake.csv")
df_true = pd.read_csv("data/True.csv")

df_fake['target'] = 0
df_true['target'] = 1
df =pd.concat([df_fake, df_true]).reset_index(drop=True)


In [3]:
# Displaying the first 5 rows of the combined dataset.
print("5 sentences of the DATA")
print(df.head())

5 sentences of the DATA
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  target  
0  December 31, 2017       0  
1  December 31, 2017       0  
2  December 30, 2017       0  
3  December 29, 2017       0  
4  December 25, 2017       0  


In [4]:
# Checking the balance between Fake and True news samples in the dataset.
print("number of samples")
print(df['target'].value_counts())

number of samples
target
0    23481
1    21417
Name: count, dtype: int64


In [5]:
# Shuffling the data and manually splitting it into 80% training and 20% testing sets.
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
train_size = int(len(df_shuffled) * 0.8)
train_df = df_shuffled.iloc[:train_size].copy()
test_df = df_shuffled.iloc[train_size:].copy()
print("\n Train Set")
print(train_df.head(5))

print("\n Test Set")
print(test_df.head(5))
print(f"train size: {len(train_df)} and test size: {len(test_df)}")


 Train Set
                                               title  \
0  Ben Stein Calls Out 9th Circuit Court: Committ...   
1  Trump drops Steve Bannon from National Securit...   
2  Puerto Rico expects U.S. to lift Jones Act shi...   
3   OOPS: Trump Just Accidentally Confirmed He Le...   
4  Donald Trump heads for Scotland to reopen a go...   

                                                text       subject  \
0  21st Century Wire says Ben Stein, reputable pr...       US_News   
1  WASHINGTON (Reuters) - U.S. President Donald T...  politicsNews   
2  (Reuters) - Puerto Rico Governor Ricardo Rosse...  politicsNews   
3  On Monday, Donald Trump once again embarrassed...          News   
4  GLASGOW, Scotland (Reuters) - Most U.S. presid...  politicsNews   

                  date  target  
0    February 13, 2017       0  
1       April 5, 2017        1  
2  September 27, 2017        1  
3         May 22, 2017       0  
4       June 24, 2016        1  

 Test Set
                     

In [6]:
# Defining and applying a function to clean the text (lowercasing, removing URLs, punctuation, and numbers).

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'^.*? - ', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

train_df['clean_text'] = train_df['text'].apply(preprocess_text)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)

In [7]:
# Printing examples to compare the original text with the cleaned text.
for i in range(2):
    original = train_df['text'].iloc[i][:150]
    cleaned = train_df['clean_text'].iloc[i][:150]

    print(f"--- Example {i+1} ---")
    print(f"Original: {original}...")
    print(f"Cleaned:  {cleaned}...")
    print("-" * 30)


--- Example 1 ---
Original: 21st Century Wire says Ben Stein, reputable professor from, Pepperdine University (also of some Hollywood fame appearing in TV shows and films such as...
Cleaned:   century wire says ben stein reputable professor from pepperdine university also of some hollywood fame appearing in tv shows and films such as ferris...
------------------------------
--- Example 2 ---
Original: WASHINGTON (Reuters) - U.S. President Donald Trump removed his chief strategist Steve Bannon from the National Security Council on Wednesday, reversin...
Cleaned:  us president donald trump removed his chief strategist steve bannon from the national security council on wednesday reversing his controversial decisi...
------------------------------


In [8]:
# Building a Bag of Words (BoW) vocabulary (max 3000 words) and converting text into numerical feature vectors.
def build_vocab(texts, max_features=3000):
    word_counts = {}
    for text in texts:
        for word in text.split():
            word_counts[word] = word_counts.get(word, 0) + 1

    sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
    vocab = {word: i for i, (word, count) in enumerate(sorted_words[:max_features])}

    print(f"Vocabulary built with {len(vocab)} features.")
    return vocab

def transform_to_bow(texts, vocab):
    features = np.zeros((len(texts), len(vocab)))
    for i, text in enumerate(texts):
        for word in text.split():
            if word in vocab:
                features[i, vocab[word]] += 1
    return features

# עכשיו זה יעבוד בלי לקרוס:
vocab = build_vocab(train_df['clean_text'], max_features=3000)
X_train = transform_to_bow(train_df['clean_text'], vocab)
X_test = transform_to_bow(test_df['clean_text'], vocab)

Vocabulary built with 3000 features.


In [9]:
# Printing the first 10 numerical features of the BoW vectors for train/test samples.
print("Feature Engineering Examples:")
print("train")
for i in range(2):
    print(f"Sample {i+1} BoW (first 10 features): {X_train[i][:10]}")
print("-" * 30)
print("test")
for i in range(2):
    print(f"Sample {i+1} BoW (first 10 features): {X_test[i][:10]}")
print("-" * 30)

Feature Engineering Examples:
train
Sample 1 BoW (first 10 features): [17.  4.  4.  2.  5.  5.  2.  4.  3.  1.]
Sample 2 BoW (first 10 features): [47. 21. 20. 20. 17. 17.  6. 15.  0.  6.]
------------------------------
test
Sample 1 BoW (first 10 features): [38. 15. 12.  9.  8.  8.  7.  1.  5.  1.]
Sample 2 BoW (first 10 features): [30. 18.  6. 12. 13. 10.  3.  2.  8.  4.]
------------------------------


In [10]:
# Defining a custom Logistic Regression model using Gradient Descent for training.
# Includes forward pass (sigmoid) and backward pass (gradient calculation).
class MyLogisticRegression:
    def __init__(self, learning_rate=0.5, epochs=1000):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def train(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for epoch in range(self.epochs):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self._sigmoid(linear_model)

            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if epoch % 100 == 0:
                print(f"Epoch {epoch}/{self.epochs} completed")

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self._sigmoid(linear_model)
        return np.array([1 if i > 0.5 else 0 for i in y_predicted])

In [11]:
# Defining a custom F1-score function.
# Training the final model with the best learning rate and evaluating its performance on the test set.
def calculate_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1)) # True Positives
    fp = np.sum((y_true == 0) & (y_pred == 1)) # False Positives
    fn = np.sum((y_true == 1) & (y_pred == 0)) # False Negatives

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return f1

print("Starting training process...")
model = MyLogisticRegression(learning_rate=0.7, epochs=500)
model.train(X_train, train_df['target'].values)

predictions = model.predict(X_test)
y_test_actual = test_df['target'].values

print("\n--- Model Evaluation ---")
print(f"First 5 Predictions: {predictions[:5]}")
print(f"First 5 Actual:      {y_test_actual[:5]}")

score = calculate_f1(y_test_actual, predictions)
print(f"Model F1-Score: {score:.4f}")

Starting training process...
Epoch 0/500 completed
Epoch 100/500 completed
Epoch 200/500 completed
Epoch 300/500 completed
Epoch 400/500 completed

--- Model Evaluation ---
First 5 Predictions: [0 0 0 0 0]
First 5 Actual:      [0 0 0 0 0]
Model F1-Score: 0.9741


In [12]:
# Implementing K-Fold Cross Validation (k=5) to evaluate different learning rates and find the optimal one.
def k_fold_cross_validation(X, y, learning_rates, k=5):
    results = []

    indices = np.arange(len(X))
    fold_sizes = np.array_split(indices, k)

    for lr in learning_rates:
        fold_scores = []
        print(f"Testing Learning Rate: {lr}")

        for i in range(k):
            val_indices = fold_sizes[i]
            train_indices = np.concatenate([fold_sizes[j] for j in range(k) if j != i])

            X_train_fold, X_val_fold = X[train_indices], X[val_indices]
            y_train_fold, y_val_fold = y[train_indices], y[val_indices]

            model = MyLogisticRegression(learning_rate=lr, epochs=300)
            model.train(X_train_fold, y_train_fold)

            preds = model.predict(X_val_fold)
            score = calculate_f1(y_val_fold, preds)
            fold_scores.append(score)

        avg_score = np.mean(fold_scores)
        results.append({
            'learning_rate': lr,
            'avg_f1_score': avg_score,
            'fold_scores': fold_scores
        })

    return pd.DataFrame(results)

lrs_to_test = [0.1, 0.5, 1.0]
cv_results_df = k_fold_cross_validation(X_train, train_df['target'].values, lrs_to_test)

print("\n--- Part 6: Grid Search & K-Fold Results ---")
print(cv_results_df)

best_lr = cv_results_df.loc[cv_results_df['avg_f1_score'].idxmax()]['learning_rate']
print(f"\nBest Hyperparameter: learning_rate = {best_lr}")

Testing Learning Rate: 0.1
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Testing Learning Rate: 0.5
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Testing Learning Rate: 1.0
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/300 completed
Epoch 100/300 completed
Epoch 200/300 completed
Epoch 0/3